<a href="https://colab.research.google.com/github/NBall65097/Fantasy-Story-Weaver/blob/main/Fantasy_Story_Weaver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers==0.0.28.post2 trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-jae6fnvo/unsloth_7800a472d4544981827cc260a39646c0
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-jae6fnvo/unsloth_7800a472d4544981827cc260a39646c0
  Resolved https://github.com/unslothai/unsloth.git to commit 6d83ad9a2834fc84cb39bd2a2fe2ba0ceb8d8262
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
from huggingface_hub import login
from google.colab import userdata
token = userdata.get('HuggingFaceWrite')
login(token) # Connect to HuggingFace account

In [3]:
import datasets # datasets function for importing data

In [4]:
# # This was used to debug issues with the AI-generated training data formatting.

# import json

# file_path = '/content/data/FSTData.jsonl'

# with open(file_path, 'r', encoding='utf-8') as f:
#     for i, line in enumerate(f, 1):
#         line = line.strip()
#         if not line:
#             continue
#         try:
#             json.loads(line)
#         except json.JSONDecodeError as e:
#             print(f"First error at row {i}: {e}")
#             print(f"Problem line (first 200 chars): {line[:200]}...")
#             print(f"Full line length: {len(line)}")
#             break
#     else:
#         print("All lines appear valid!")

In [5]:
#filepath = "/content/data/FSTData.jsonl"
#data = datasets.load_dataset("json",data_files=filepath,split='train')
#data.push_to_hub('NBall65097/fantasy-storyweaver-data') # Save dataset to HuggingFace

In [7]:
import torch
from datasets import load_dataset
import unsloth
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [8]:
model, tokenizer = FastLanguageModel.from_pretrained(
    "microsoft/Phi-3.5-mini-instruct",
    dtype=None,  # auto
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.3.17: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [9]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

Unsloth 2026.3.17 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [20]:
import json
dataset = load_dataset('NBall65097/fantasy-storyweaver-data',split="train")

In [21]:
from unsloth.chat_templates import apply_chat_template

In [22]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="phi-3")

def format_phi35_data(examples):
  formatted_texts = []
  for msg_list in examples["messages"]:
    text = tokenizer.apply_chat_template(
        msg_list,
        tokenize=False,
        add_generation_prompt=False,
    )
    formatted_texts.append(text)
  return {"text":formatted_texts}

In [23]:
dataset = dataset.map(
    format_phi35_data,
    batched=True,
    batch_size=1000,          # adjust based on RAM
    num_proc=2,               # or 4 if you have enough CPU cores
    desc="Formatting Phi-3.5 chat examples",
)

Formatting Phi-3.5 chat examples (num_proc=2):   0%|          | 0/165 [00:00<?, ? examples/s]

In [24]:
data = dataset.train_test_split(test_size=0.1, train_size=0.9)
traindata, testdata = data["train"], data["test"]

In [28]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=traindata,
    eval_dataset=testdata,
    args=SFTConfig(
        output_dir="/content/output",
        num_train_epochs=2,
        learning_rate=2e-4,
        max_seq_length=2048,
        dataset_text_field="text",  # or use formatting_func for chat template
        packing=True,  # packs multiple examples → huge speed win
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_ratio=0.03,
        fp16=True,
        gradient_checkpointing="unsloth",
        optim="adamw_8bit",
        logging_steps=10,
        save_strategy="epoch",
        #save_steps=200,
        report_to="none",
    )  # TrainingArguments: 1–3 epochs, lr=2e-4, batch=4–8 (gradient accum=4), warmup, etc.
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Packing train dataset (num_proc=4):   0%|          | 0/148 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/17 [00:00<?, ? examples/s]

Unsloth: Packing eval dataset (num_proc=4):   0%|          | 0/17 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


In [29]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 43 | Num Epochs = 2 | Total steps = 6
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 29,884,416 of 3,850,963,968 (0.78% trained)


Step,Training Loss


Step,Training Loss


TrainOutput(global_step=6, training_loss=1.5512415568033855, metrics={'train_runtime': 362.873, 'train_samples_per_second': 0.237, 'train_steps_per_second': 0.017, 'total_flos': 3506451862118400.0, 'train_loss': 1.5512415568033855, 'epoch': 2.0})

In [30]:
model.save_pretrained("fantasy-story-weaver-lora")          # saves adapter only (~50–200 MB)
tokenizer.save_pretrained("fantasy-story-weaver-lora")

('fantasy-story-weaver-lora/tokenizer_config.json',
 'fantasy-story-weaver-lora/chat_template.jinja',
 'fantasy-story-weaver-lora/tokenizer.json')

In [31]:
model.save_pretrained_merged(
    "fantasy-story-weaver-merged",
    tokenizer,
    save_method = "merged_16bit",
)

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:00<00:00, 14122.24it/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:10<00:00, 65.15s/it]


Unsloth: Merge process complete. Saved to `/content/fantasy-story-weaver-merged`


In [32]:
# # This just pushes the LoRA adapter. Small, fast, but not the full model.

# model.push_to_hub("NBall65097/fantasy-story-weaver-lora")
# tokenizer.push_to_hub("NBall65097/fantasy-story-weaver-lora")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 29.3kB /  120MB            

Saved model to https://huggingface.co/NBall65097/fantasy-story-weaver-lora


No files have been modified since last commit. Skipping to prevent empty commit.


In [33]:
# # This pushes the full, merged model.

# model.push_to_hub_merged(
#     "NBall65097/fantasy-story-weaver",
#     tokenizer,
#     save_method = "merged_16bit",
#     token = True
# )

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

No files have been modified since last commit. Skipping to prevent empty commit.


Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0002-of-00002.safetensors:   5%|4         |  128MB / 2.65GB            


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [06:07<00:00, 183.94s/it]


Unsloth: Merge process complete. Saved to `/content/NBall65097/fantasy-story-weaver`


**Evaluation**

In [34]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    "NBall65097/fantasy-story-weaver",
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

eval_results = trainer.evaluate()
print(eval_results)

==((====))==  Unsloth 2026.3.17: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/849 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.jinja:   0%|          | 0.00/371 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32009)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Llam